# 01 · Dataset profile: Kottarathil PCOS dataset

**Tracker task:** IMP-02 · Load and inspect the Kottarathil 541-record dataset

**Evidence required:** record count, feature count, class balance.

In [1]:
from pathlib import Path

import pandas as pd

# Project root: works whether the notebook is started from notebooks/ or from the project folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "raw" / "PCOS_data_without_infertility.xlsx"

pd.set_option("display.max_rows", 60)
print("Reading:", DATA_FILE.relative_to(ROOT))

Reading: data\raw\PCOS_data_without_infertility.xlsx


## 1. Sheets in the file

The Excel file has more than one sheet. The data is in `Full_new`; `Instructions` describes how the data was coded.

In [2]:
xls = pd.ExcelFile(DATA_FILE)
xls.sheet_names

['Instructions', 'Full_new']

## 2. Load the data

In [3]:
df = pd.read_excel(DATA_FILE, sheet_name="Full_new")
n_records, n_columns = df.shape
print(f"Records: {n_records}")
print(f"Columns: {n_columns}")

Records: 541
Columns: 45


## 3. Column inventory

Every column with its type, how many values it has, and an example. Column names are shown with quotes (`repr`) so hidden leading or trailing spaces are visible.

Each column is given a **role**:
- **identifier:** row or patient number. Never used as a model input.
- **label:** what the model predicts (`PCOS (Y/N)`).
- **stray:** an unnamed column that is not part of the data.
- **feature:** everything else, i.e. potential model inputs.

In [4]:
ID_COLS = ["Sl. No", "Patient File No."]
LABEL = "PCOS (Y/N)"
STRAY = [c for c in df.columns if str(c).startswith("Unnamed")]

def role(col):
    if col in ID_COLS:
        return "identifier"
    if col == LABEL:
        return "label"
    if col in STRAY:
        return "stray"
    return "feature"

inventory = pd.DataFrame({
    "column": [repr(c) for c in df.columns],
    "role": [role(c) for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "non_missing": df.notna().sum().values,
    "unique_values": df.nunique().values,
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
inventory

,column,role,dtype,non_missing,unique_values,example
0,'Sl. No',identifier,int64,541,541,1
1,'Patient File No.',identifier,int64,541,541,1
2,'PCOS (Y/N)',label,int64,541,2,0
3,' Age (yrs)',feature,int64,541,29,28
4,'Weight (Kg)',feature,float64,541,117,44.6
5,'Height(Cm) ',feature,float64,541,50,152.0
6,'BMI',feature,float64,541,355,19.3
7,'Blood Group',feature,int64,541,8,15
8,'Pulse rate(bpm) ',feature,int64,541,11,78
9,'RR (breaths/min)',feature,int64,541,8,22


## 4. Feature count

In [5]:
role_counts = inventory["role"].value_counts().reindex(["identifier", "label", "feature", "stray"], fill_value=0)
role_counts.to_frame("columns")

,columns
role,
identifier,2
label,1
feature,41
stray,1


## 5. Class balance

In [6]:
counts = df[LABEL].value_counts().sort_index()
balance = pd.DataFrame({
    "class": ["Not PCOS (0)", "PCOS (1)"],
    "records": counts.values,
    "percent": (100 * counts / counts.sum()).round(1).values,
})
majority = balance["percent"].max()
print(f"A model that always predicts the majority class would score {majority}% accuracy.")
balance

A model that always predicts the majority class would score 67.3% accuracy.


,class,records,percent
0,Not PCOS (0),364,67.3
1,PCOS (1),177,32.7


## 6. Summary (IMP-02 evidence)

| Item | Value |
|---|---|
| Records | 541 |
| Columns in file | 45 |
| Identifiers / label / stray | 2 / 1 / 1 |
| **Features** | **41** |
| **Class balance** | **177 PCOS (32.7%) vs 364 not PCOS (67.3%)** |

**Observations to carry forward**
- The classes are moderately imbalanced, so accuracy alone is misleading (67.3% by always predicting "not PCOS"). Use stratified cross-validation and report recall, specificity, F1, ROC-AUC and PR-AUC (IMP-07).
- Several column names have leading/trailing spaces (e.g. `' Age (yrs)'`, `'Height(Cm) '`), and `No. of aborptions` is misspelled. Column names should be standardised in code before modelling.
- `AMH(ng/mL)` and `II beta-HCG(mIU/mL)` are stored as text (`object`) although they should be numeric, which suggests some non-numeric entries. Investigate in IMP-04.
- `Marraige Status (Yrs)` and `Fast food (Y/N)` each have one missing value. Full missing-value analysis is IMP-03.
- `Unnamed: 44` holds only 2 values and is not a real data column.